In [0]:
# Cell 1: JSON arrays -> one row per hour -> MERGE into silver.weather_hourly
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType

CATALOG = "city_delta"
BRONZE_TABLE = f"{CATALOG}.bronze.weather_raw"
SILVER_TABLE = f"{CATALOG}.silver.weather_hourly"

hourly_schema = StructType([
    StructField("hourly", StructType([
        StructField("time",                      ArrayType(StringType())),
        StructField("temperature_2m",            ArrayType(DoubleType())),
        StructField("precipitation_probability", ArrayType(IntegerType())),
        StructField("precipitation",             ArrayType(DoubleType())),
        StructField("weather_code",              ArrayType(IntegerType())),
        StructField("wind_speed_10m",            ArrayType(DoubleType())),
    ]))
])

latest = spark.sql(f"SELECT max(ingest_date) AS d FROM {BRONZE_TABLE}").first()["d"]

h = F.from_json(col("raw_json"), hourly_schema)["hourly"]

weather = (
    spark.table(BRONZE_TABLE)
    .where(col("ingest_date") == latest)
    .select("city_key", col("ingest_ts").alias("forecast_made_ts"), h.alias("h"))
    .select("city_key", "forecast_made_ts",
            F.explode(F.arrays_zip(
                col("h.time").alias("time"),
                col("h.temperature_2m").alias("temperature_c"),
                col("h.precipitation_probability").alias("rain_chance_pct"),
                col("h.precipitation").alias("rain_mm"),
                col("h.weather_code").alias("weather_code"),
                col("h.wind_speed_10m").alias("wind_kmh"),
            )).alias("x"))
    .select("city_key", "forecast_made_ts", "x.*")
    .withColumn("hour_local", col("time").cast("timestamp_ntz"))   # Amsterdam local time, no timezone shift
    .withColumn("date_local", F.to_date("hour_local"))
    .withColumn("rain_risk",  col("rain_chance_pct") >= 60)        # roadmap: badge above 60 %
    .drop("time")
)

if not spark.catalog.tableExists(SILVER_TABLE):
    weather.limit(0).write.format("delta").saveAsTable(SILVER_TABLE)

weather.createOrReplaceTempView("weather_updates")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS t
    USING weather_updates AS s
      ON t.city_key = s.city_key AND t.hour_local = s.hour_local
    WHEN MATCHED AND s.forecast_made_ts >= t.forecast_made_ts THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"""
    SELECT date_local,
           round(min(temperature_c), 1) AS min_c,
           round(max(temperature_c), 1) AS max_c,
           max(rain_chance_pct)         AS max_rain_pct,
           sum(CASE WHEN rain_risk THEN 1 ELSE 0 END) AS rainy_hours
    FROM {SILVER_TABLE}
    GROUP BY date_local
    ORDER BY date_local
"""))